# Unified Local LLM Demo Cases

This notebook demonstrates the final model-handle API:

```python
server = LLMProviderPool()
llm = server.load_model(provider, model, defaults...)
response = await llm.call(messages=[...])
```

Provider config stays endpoint-only. Model ids and inference defaults live on `LocalLLM` handles. Structured-output repair, loop guard, tools, and logging stay as separate layers.

## Imports And Provider Registry

Use `providers.example.json` as a starting point, or create `providers.json` with the same shape. Provider keys should be canonical: `ollama`, `lm_studio`, `unsloth`, `llama_cpp`.

In [5]:
from pathlib import Path
import json
import requests

from unified_local_llm_server import LLMProviderPool
from unified_local_llm_server.llm_logger import AsyncLLMLogger
from unified_local_llm_server.provider_registry import ProviderRegistry
from unified_local_llm_server.helpers.pydantic_helper import dict_to_pydantic_schema

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROVIDERS_PATH = ROOT / "providers.example.yaml"
LOG_DIR = ROOT / "logs"
LOG_DIR.mkdir(exist_ok=True)

registry = ProviderRegistry.load(PROVIDERS_PATH)
server = LLMProviderPool(provider_registry=registry)
providers_names = server.get_providers()
provider_status = {}
for provider_name in server.get_providers():
    provider_status[provider_name] = await server.check_provider(provider_name)

for pvd_nm in providers_names:
    provider_data = (await server.check_provider(pvd_nm))
    print(provider_data)
    if provider_data['ok']:
        for model in server.list_downloaded_models(pvd_nm):
            print(f"Provider: {pvd_nm} - Model: {model}")

{'provider': 'llama_cpp', 'server_url': 'http://127.0.0.1:9090', 'ok': True, 'kind': 'health', 'data': {'provider': 'llama_cpp', 'server_url': 'http://127.0.0.1:8080', 'ok': False, 'error': 'llama-server not running'}}
Provider: llama_cpp - Model: unsloth_gpt-oss-120b-GGUF_UD-Q8_K_XL_gpt-oss-120b-UD-Q8_K_XL
Provider: llama_cpp - Model: DeepSeek-R1-0528-Qwen3-8B-Q4_K_M
Provider: llama_cpp - Model: gemma-4-E4B-it-Q4_K_M
Provider: llama_cpp - Model: gpt-oss-20b-MXFP4
Provider: llama_cpp - Model: Qwen3.6-27B-UD-Q3_K_XL
Provider: llama_cpp - Model: Qwen3.6-27B-UD-Q4_K_XL
Provider: llama_cpp - Model: gemma-4-26B-A4B-it-UD-Q4_K_XL
Provider: llama_cpp - Model: gemma-4-26B-A4B-it-UD-Q4_K_M
Provider: llama_cpp - Model: gemma-4-31B-it-UD-Q4_K_XL
Provider: llama_cpp - Model: gemma-4-E2B-it-UD-Q4_K_XL
Provider: llama_cpp - Model: gpt-oss-20b-UD-Q4_K_XL
{'provider': 'lm_studio', 'server_url': 'http://127.0.0.1:1234', 'ok': True, 'kind': 'models', 'data': {'data': [{'id': '90f9618340396838ee7ff5b0ba2

In [2]:
server.unload_all_models(None)

llm0 = server.load_model("llama_cpp", "gpt-oss-20b-MXFP4")

## Model Choices

Edit these values to match your downloaded or currently loaded local models. `None` means the server can use the first id returned from `/v1/models` when that provider supports it.

In [3]:
simple_response = await llm0.call(
    messages=[
        {"role": "system", "content": "Answer in one concise sentence."},
        {"role": "user", "content": "Why do local LLM interfaces need provider abstraction?"},
    ],
    options={"num_predict": 128},
)
simple_response

'We need to explain why local LLM interfaces need provider abstraction. Likely because local LLMs might have different backends, APIs, frameworks like Hugging Face, Llama.cpp, exllamav2, llama.cpp, etc. Provider abstraction allows uniform interface, easier switching, modularity, plugin architecture. It also separates concerns: inference engine vs model. Provide benefits like code reuse, easier maintenance, ability to plug in new models without rewriting UI. Also local LLMs may need different tokenizers, quantization, GPU/CPU options.\n\nNeed to answer succinctly but thorough.'

In [4]:
server.unload_all_models(None)

{'llama_cpp': {'unloaded': ['gpt-oss-20b-MXFP4.gguf']},
 'lm_studio': {'unloaded': []},
 'ollama': {'unloaded': []},
 'unsloth': {'unloaded': []}}

## Provider Readiness

This checks provider ports and model-list endpoints. It does not run inference.

In [ ]:
server.list_downloaded_models("lm_studio")

In [ ]:
server.unload_all_models("lm_studio")

In [ ]:
server.list_loaded_models("lm_studio")

In [ ]:
llm0 = server.load_model("lm_studio", "deepseek/deepseek-r1-0528-qwen3-8b")

In [ ]:
simple_response = await llm0.call(
    messages=[
        {"role": "system", "content": "Answer in one concise sentence."},
        {"role": "user", "content": "Why do local LLM interfaces need provider abstraction?"},
    ],
    options={"num_predict": 128},
)

In [ ]:
simple_response

## List Models For Each Available Provider

This calls every provider's configured `models_path`, usually `/v1/models`, and normalizes the output into rows with endpoint details and model ids.

In [ ]:
def extract_model_ids(models_response):
    if not isinstance(models_response, dict):
        return []
    data = models_response.get("data")
    if not isinstance(data, list):
        return []
    return [item.get("id") for item in data if isinstance(item, dict) and item.get("id")]

models_by_provider = {}
model_listing_rows = []

for provider_name in server.get_providers():
    provider_server = server.get_provider(provider_name)
    status = provider_status.get(provider_name) or await provider_server.check_provider()
    row = {
        "provider": provider_name,
        "server_url": provider_server.server_url,
        "base_url": provider_server.base_url,
        "models_path": provider_server.config.models_path,
        "ok": bool(status.get("ok")),
        "model_ids": [],
        "error": None,
    }

    if not row["ok"]:
        row["error"] = status.get("error")
        models_by_provider[provider_name] = {"error": row["error"]}
        model_listing_rows.append(row)
        continue

    try:
        models_response = await provider_server.list_models()
        models_by_provider[provider_name] = models_response
        row["model_ids"] = extract_model_ids(models_response)
    except Exception as exc:
        row["ok"] = False
        row["error"] = str(exc)
        models_by_provider[provider_name] = {"error": row["error"]}

    model_listing_rows.append(row)

model_listing_rows

## Resolve Model Ids From Live Listings

Configured model ids are examples. This cell checks each provider's `/v1/models` output and chooses an available model when the preferred id is missing. Embedding models are skipped for chat demos when possible.

In [ ]:
def choose_chat_model(provider_name: str, preferred: str | None, rows: list[dict]) -> str | None:
    row = next((item for item in rows if item["provider"] == provider_name), None)
    model_ids = [] if row is None else list(row.get("model_ids") or [])
    if preferred and (not model_ids or preferred in model_ids):
        return preferred
    chat_like = [model_id for model_id in model_ids if "embed" not in model_id.lower()]
    if chat_like:
        return chat_like[0]
    return model_ids[0] if model_ids else preferred

MODEL_BY_PROVIDER = {
    provider_name: choose_chat_model(
        provider_name,
        PREFERRED_MODEL_BY_PROVIDER.get(provider_name),
        model_listing_rows,
    )
    for provider_name in server.get_providers()
}

MODEL_BY_PROVIDER

## Optional Unsloth Model Load

Unsloth has provider-management endpoints outside the OpenAI `/v1` API. Keep this separate from `LocalLLM.call()`.

In [ ]:
# Edit before running.
RUN_UNSLOTH_LOAD = False
UNSLOTH_PORT = 8899
UNSLOTH_KEY = "replace-with-unsloth-key"
UNSLOTH_MODEL_PATH = MODEL_BY_PROVIDER["unsloth"]

if RUN_UNSLOTH_LOAD:
    auth_hdr = {"Authorization": f"Bearer {UNSLOTH_KEY}"}
    response = requests.post(
        f"http://127.0.0.1:{UNSLOTH_PORT}/api/inference/load",
        headers=auth_hdr,
        json={"model_path": UNSLOTH_MODEL_PATH},
        timeout=300,
    )
    response.raise_for_status()
    unsloth_load_result = response.json()
else:
    unsloth_load_result = {"skipped": True}

unsloth_load_result

## Load Model Handles

Each handle binds provider endpoint, model id, and inference defaults. This is the primary API for experiments.

In [ ]:
llm_ollama = server.load_model(
    "ollama",
    MODEL_BY_PROVIDER["ollama"] or "replace-with-ollama-model-id",
    temperature=0.2,
    context_length=8192,
    options=DEFAULT_OPTIONS["ollama"],
)

llm_lmstudio = server.load_model(
    "lm_studio",
    MODEL_BY_PROVIDER["lm_studio"] or "replace-with-lm-studio-model-id",
    temperature=0.1,
    options=DEFAULT_OPTIONS["lm_studio"],
)

llm_unsloth = server.load_model(
    "unsloth",
    MODEL_BY_PROVIDER["unsloth"] or "replace-with-unsloth-model-id",
    temperature=0.1,
    options=DEFAULT_OPTIONS["unsloth"],
)

llm_llama_cpp = server.load_model(
    "llama_cpp",
    MODEL_BY_PROVIDER["llama_cpp"] or "replace-with-llama-cpp-model-id",
    temperature=0.2,
    options=DEFAULT_OPTIONS["llama_cpp"],
)

LLMS = {
    "ollama": llm_ollama,
    "lm_studio": llm_lmstudio,
    "unsloth": llm_unsloth,
    "llama_cpp": llm_llama_cpp,
}

{key: {"provider": llm.provider, "model": llm.model, "base_url": llm.base_url} for key, llm in LLMS.items()}

## Simple Chat Generation

Plain text calls do not use JSON-fix retry settings. Loop guard uses `max_loop_retries`, default `5`.

In [ ]:
active_provider = next(
    (
        provider_name
        for provider_name, status in provider_status.items()
        if status.get("ok") and MODEL_BY_PROVIDER.get(provider_name)
    ),
    "ollama",
)
active = LLMS[active_provider]

simple_response = await active.call(
    messages=[
        {"role": "system", "content": "Answer in one concise sentence."},
        {"role": "user", "content": "Why do local LLM interfaces need provider abstraction?"},
    ],
    options={"num_predict": 128},
)

{"provider": active_provider, "model": active.model, "response": simple_response}

## Structured Output

`schema_dict` activates the JSON-fix layer. `max_json_fix_retries` belongs only to this structured-output path.

In [ ]:
schema_dict = {
    "provider": str,
    "model": str,
    "works": bool,
    "risks": [str],
}

schema_json = dict_to_pydantic_schema(schema_dict).model_json_schema()
schema_json

In [ ]:
structured_response = await active.call(
    messages=[
        {"role": "system", "content": "Return only JSON matching the requested schema."},
        {"role": "user", "content": "Describe this local LLM call as a status object."},
    ],
    schema_dict=schema_dict,
    temperature=0.1,
    options={"num_predict": 512},
    max_json_fix_retries=3,
)

structured_response

## Compare Providers

This runs the same unstructured prompt across reachable providers. Errors are captured per provider so one failed local server does not stop the whole comparison.

In [ ]:
comparison_prompt = "Return exactly five words about robust local inference."
comparison_results = {}

for provider_name, llm in LLMS.items():
    status = provider_status.get(provider_name, {})
    if not status.get("ok"):
        comparison_results[provider_name] = {"skipped": status.get("error")}
        continue
    try:
        comparison_results[provider_name] = await llm.call(
            messages=[{"role": "user", "content": comparison_prompt}],
            temperature=0.3,
            options={"num_predict": 128},
        )
    except Exception as exc:
        comparison_results[provider_name] = {"error": str(exc)}

comparison_results

## Multiple Models On One Provider

Use separate handles for different loaded models on the same provider. Provider endpoint stays the same; model identity changes on the handle.

In [ ]:
lmstudio_model_a = server.load_model(
    "lm_studio",
    MODEL_BY_PROVIDER["lm_studio"],
    temperature=0.1,
    options={"num_predict": 128},
)

lmstudio_model_b = server.load_model(
    "lm_studio",
    "replace-with-second-lm-studio-model-id",
    temperature=0.4,
    options={"num_predict": 128},
)

# Run after replacing model_b with a real id.
# response_a = await lmstudio_model_a.call(messages=[{"role": "user", "content": "Name your model family."}])
# response_b = await lmstudio_model_b.call(messages=[{"role": "user", "content": "Name your model family."}])
# {"a": response_a, "b": response_b}

## Batch Calls

Batch belongs to `LocalLLM`, not `LLMProviderPool`, because most batch experiments target one model handle with fixed defaults.

Supported item shapes:

```python
list[list[message]]
list[dict]  # per-item call kwargs, must include messages
```


In [ ]:
simple_batch_items = [
    [{"role": "user", "content": "Return one adjective for local models."}],
    [{"role": "user", "content": "Return one risk of weak local models."}],
    [{"role": "user", "content": "Return one benefit of provider abstraction."}],
]

simple_batch_results = await active.batch(
    simple_batch_items,
    concurrency=2,
    return_exceptions=True,
    options={"num_predict": 96},
)

simple_batch_results

## Batch With Per-Item Overrides

Each dict item is passed to `llm.call(...)`. Shared kwargs apply to every item unless the item overrides them. `options` are merged.

In [ ]:
override_batch_items = [
    {
        "messages": [{"role": "user", "content": "Return one short sentence about JSON reliability."}],
        "temperature": 0.1,
        "options": {"num_predict": 96},
    },
    {
        "messages": [{"role": "user", "content": "Return one creative sentence about local inference."}],
        "temperature": 0.7,
        "options": {"num_predict": 128},
    },
]

override_batch_results = await active.batch(
    override_batch_items,
    concurrency=2,
    return_exceptions=True,
    options={"top_p": 0.9},
)

override_batch_results

## Structured Batch

Structured batch uses the same JSON-fix layer as `call(...)`. `max_json_fix_retries` is shared here, and each item can still override prompt, temperature, or options.

In [ ]:
batch_schema = {
    "label": str,
    "score": float,
    "notes": [str],
}

structured_batch_items = [
    {
        "messages": [{"role": "user", "content": "Score Ollama as a local experimentation backend."}],
        "temperature": 0.1,
    },
    {
        "messages": [{"role": "user", "content": "Score LM Studio as a local experimentation backend."}],
        "temperature": 0.1,
    },
]

structured_batch_results = await active.batch(
    structured_batch_items,
    concurrency=2,
    return_exceptions=True,
    schema_dict=batch_schema,
    max_json_fix_retries=3,
    options={"num_predict": 512},
)

structured_batch_results

## Loop Guard Stress

Weak models sometimes repeat. `max_loop_retries` is separate from JSON repair and defaults to `5`.

In [ ]:
loop_stress_result = await active.call(
    messages=[
        {"role": "system", "content": "Answer normally. Do not repeat phrases."},
        {"role": "user", "content": "Write a short note about avoiding repeated outputs."},
    ],
    temperature=0.8,
    options={"num_predict": 512},
    max_loop_retries=5,
)

loop_stress_result

## Tool Pipeline Demo

The tool layer owns tool-call execution. The model must support OpenAI-style tool calls for this to run end-to-end.

In [ ]:
def add_tool(arguments: dict) -> dict:
    return {"sum": arguments["a"] + arguments["b"]}

tools = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Add two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    }
]

# Run only with a model/provider that emits OpenAI-compatible tool_calls.
# tool_result = await active.call(
#     messages=[{"role": "user", "content": "Use the add tool for 2 + 3, then answer."}],
#     tools=tools,
#     tool_registry={"add": add_tool},
#     max_tool_rounds=3,
# )
# tool_result

## Async Logging

The logger records request, response, parser errors, parsed output, and tool results in a readable stream-style log.

In [ ]:
log_path = LOG_DIR / "demo_llm_call.log"
logger = AsyncLLMLogger(log_path)
logged = server.load_model(
    "ollama",
    MODEL_BY_PROVIDER["ollama"],
    logger=logger,
    temperature=0.2,
    options={"num_predict": 128},
)

logged_response = await logged.call(
    messages=[{"role": "user", "content": "Say one reason to log local LLM responses."}],
)
await logged.server.close()

logged_response

In [ ]:
log_path.read_text(encoding="utf-8").splitlines()[:12]

## Direct OpenAI-Compatible Client

`LLMProviderPool.openai_async_client()` is still available when you need direct OpenAI-style access. Pipelines are bypassed here.

In [ ]:
raw_server = server.get_provider("ollama")
raw_server.model = MODEL_BY_PROVIDER["ollama"]
client = raw_server.openai_async_client()

# response = await client.chat.completions.create(
#     model=MODEL_BY_PROVIDER["ollama"],
#     messages=[{"role": "user", "content": "Return a raw OpenAI-compatible response."}],
# )
# response

## Structured Stress Case

This intentionally asks for structured output from a weak local model. Use it to inspect JSON-fix retries and logs.

In [ ]:
stress_schema = {
    "summary": str,
    "confidence": float,
    "failure_modes": [str],
}

stress_result = await active.call(
    messages=[
        {"role": "system", "content": "Return only JSON. No markdown."},
        {"role": "user", "content": "Summarize likely failure modes for weak local structured output."},
    ],
    schema_dict=stress_schema,
    temperature=0.7,
    options={"num_predict": 768},
    max_json_fix_retries=3,
    max_loop_retries=5,
)

stress_result